# 📰 Tarea 1 – Parte 1: Recolección de noticias de El Comercio

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

| Campo | Valor |
|---|---|
| **Opción elegida** | **B – El Comercio** (350–400 noticias, ~90–100 por categoría) |
| **URL base** | https://elcomercio.pe |
| **robots.txt** | https://elcomercio.pe/robots.txt |
| **Fuentes usadas** | RSS oficiales (Google News feed y Arc) por sección + sitemaps web/news de cada sección declarados en robots.txt |
| **Categorías finales** | `politica`, `economia`, `deportes`, `ciencia_tecnologia` |

**Estrategia:** primero se usan los canales RSS oficiales (lo más respetuoso con el medio). Como cada RSS trae solo las noticias más recientes, se complementa con los sitemaps que el propio diario publica en su `robots.txt`. Todas las solicitudes usan un `User-Agent` descriptivo, una pausa fija de 1.5 s (`segundos_de_pausa`) y máximo 1 reintento. Cada ejecución guarda un respaldo crudo en `datos/raw/`, y la limpieza une todos los respaldos (se puede recolectar en varios días).

## 1. Configuración inicial

In [3]:
import os, re, time, random, glob
from datetime import datetime
from urllib.parse import urlparse, urlunparse
from urllib import robotparser

import requests
import pandas as pd
from bs4 import BeautifulSoup
import plotly.express as px

# Medio y parámetros de scraping respetuoso
BASE  = "https://elcomercio.pe"
MEDIO = "El Comercio"

HEADERS = {
    "User-Agent": "TareaDiplomadoPucp (proyecto academico de clasificacion de titulares)"
}
segundos_de_pausa = 1.5   # pausa entre solicitudes (el enunciado pide 1–2 s)
MAX_REINTENTOS    = 1     # máximo 1 reintento por URL
TIMEOUT           = 20    # segundos máximos de espera por respuesta

# Categorías y metas 
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
META_POR_CATEGORIA = 120      # se recolecta un poco más de 100 para compensar descartes
MAX_POR_CATEGORIA  = 100      # tope final para balancear clases

# Fecha de descarga y carpetas 
STAMP = datetime.now().strftime("%Y%m%d_%H%M")
FECHA_DESCARGA = datetime.now().strftime("%Y-%m-%d %H:%M")
os.makedirs("datos/raw", exist_ok=True)

# Gráficos: interactivos en VS Code + PNG visible en GitHub (requiere kaleido)
RENDERER = "plotly_mimetype+png"
def mostrar(fig):
    try:
        fig.show(renderer=RENDERER)
    except Exception as e:
        print("⚠️ No se pudo generar PNG (¿kaleido instalado?):", e)
        fig.show()

print("Fecha de descarga :", FECHA_DESCARGA)
print("User-Agent        :", HEADERS["User-Agent"])

Fecha de descarga : 2026-10-18 00:04
User-Agent        : TareaDiplomadoPucp (proyecto academico de clasificacion de titulares)


## 2. Revisión de `robots.txt`
Se descarga y se muestra el `robots.txt`; todas las solicitudes posteriores pasan por `permitido(url)`. Además se extraen los sitemaps que el medio declara públicamente.

In [4]:
r = requests.get(BASE + "/robots.txt", headers=HEADERS, timeout=20)
robots_txt = r.text if r.status_code == 200 else ""
print(f"robots.txt → HTTP {r.status_code}\n")
print(robots_txt[:2000])

rp = robotparser.RobotFileParser()
rp.parse(robots_txt.splitlines())

def permitido(url):
    return rp.can_fetch(HEADERS["User-Agent"], url)

SITEMAPS = re.findall(r"(?im)^\s*sitemap:\s*(\S+)", robots_txt)
print("\nSitemaps declarados:", *SITEMAPS, sep="\n  ")

robots.txt → HTTP 200

# DIRECTIVAS GENERALES
User-agent: *
Disallow: /catalogo/
Disallow: /buscar/
Disallow: /archivo/todas/19*
Disallow: /archivo/todas/200*

# BLOQUEO DE SCRAPERS ANTIGUOS Y MALICIOSOS
User-agent: UbiCrawler
User-agent: DOC
User-agent: Zao
User-agent: sitecheck.internetseer.com
User-agent: Zealbot
User-agent: MSIECrawler
User-agent: SiteSnagger
User-agent: WebStripper
User-agent: WebCopier
User-agent: Fetch
User-agent: Offline Explorer
User-agent: Teleport
User-agent: TeleportPro
User-agent: WebZIP
User-agent: linko
User-agent: HTTrack
User-agent: Microsoft.URL.Control
User-agent: Xenu
User-agent: larbin
User-agent: libwww
User-agent: ZyBORG
User-agent: Download Ninja
User-agent: wget
User-agent: grub-client
User-agent: k2spider
User-agent: NPBot
User-agent: WebReaper
Disallow: /

# SITEMAPS 
Sitemap: https://elcomercio.pe/sitemap/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/politica/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/peru/?outputType=xm

## 3. Funciones auxiliares (descarga respetuosa y parsing)

In [5]:
def descargar(url):
    """GET respetuoso: respeta robots.txt, pausa de segundos_de_pausa y máximo 1 reintento."""
    if not permitido(url):
        print("  ⛔ robots.txt no permite:", url)
        return None
    for intento in range(1 + MAX_REINTENTOS):   # 1 intento + 1 reintento
        try:
            time.sleep(segundos_de_pausa)
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code == 200:
                return resp.text
            print(f"  HTTP {resp.status_code} → {url}")
            if resp.status_code in (403, 404):   # no tiene sentido reintentar
                return None
        except requests.RequestException as e:
            print(f"  {type(e).__name__} → {url}")
    return None

def normalizar_url(u):
    p = urlparse(str(u).strip())
    return urlunparse((p.scheme or "https", p.netloc, p.path.rstrip("/") + "/", "", "", ""))

def seccion_desde_url(url):
    """https://elcomercio.pe/economia/mercados/titulo-noticia/ → 'economia-mercados'"""
    partes = [p for p in urlparse(str(url)).path.split("/") if p]
    if len(partes) > 1:
        return "-".join(partes[:-1])
    return partes[0] if partes else ""

def texto_limpio(html_o_texto):
    return BeautifulSoup(html_o_texto or "", "html.parser").get_text(" ", strip=True)

def guardar_crudo(nombre, contenido):
    with open(os.path.join("datos/raw", nombre), "w", encoding="utf-8") as f:
        f.write(contenido)

## 4. Reglas de mapeo `seccion_original` → categoría
Se definen aquí porque se usan para decidir cuántas noticias faltan por categoría. La **tabla de equivalencias real** (con conteos) se muestra en la sección de limpieza.

In [6]:
REGLAS_MAPEO = [            # (prefijo de seccion_original, categoría final)
    ("politica",   "politica"),
    ("economia",   "economia"),
    ("deporte",    "deportes"),
    ("futbol",     "deportes"),
    ("tecnologia", "ciencia_tecnologia"),
    ("ciencia",    "ciencia_tecnologia"),
]

def categoria_de(seccion):
    s = str(seccion or "").lower()
    for prefijo, cat in REGLAS_MAPEO:
        if s.startswith(prefijo):
            return cat
    return None   # no encaja en las 4 categorías → se descarta

pd.DataFrame(REGLAS_MAPEO, columns=["prefijo de seccion_original", "categoria"])

,prefijo de seccion_original,categoria
0,politica,politica
1,economia,economia
2,deporte,deportes
3,futbol,deportes
4,tecnologia,ciencia_tecnologia
5,ciencia,ciencia_tecnologia


## 5. Fuentes oficiales de cada categoría

Revisando el `robots.txt` se identificó cómo nombra El Comercio cada sección. **Deportes no está en `/deportes/`, sino en `/deporte-total/`**, y ciencia está dentro de tecnología (`/tecnologia/ciencias/`).

| Categoría final | Sección en El Comercio | Fuentes usadas (todas declaradas o públicas) |
|---|---|---|
| `politica` | `politica` | RSS Google News + RSS Arc + sitemaps web/news |
| `economia` | `economia` | RSS Google News + RSS Arc + sitemaps web/news |
| `deportes` | `deporte-total` | RSS Google News + RSS Arc + sitemaps web/news |
| `ciencia_tecnologia` | `tecnologia` (incluye `tecnologia/ciencias`) | RSS Google News + RSS Arc + sitemaps web/news |

**Problema documentado:** en la primera ejecución se usó la ruta `/deportes/`, que no existe en El Comercio, y se obtuvieron 0 noticias de deportes. Además, se recorrían los 45 sitemaps del `robots.txt` en orden, y se alcanzaba el límite antes de llegar a las secciones útiles. Se corrigió apuntando directamente a la sección de cada categoría.

In [7]:
SECCION_EC = {                     # categoría final → sección en la URL de El Comercio
    "politica":           "politica",
    "economia":           "economia",
    "deportes":           "deporte-total",
    "ciencia_tecnologia": "tecnologia",
}

FEEDS_RSS = [                      # canales RSS oficiales por sección
    BASE + "/arc/outboundfeeds/google-news-feed/category/{s}/?outputType=xml",
    BASE + "/arcio/rss/category/{s}/",
]
SITEMAPS_SECCION = [               # sitemaps declarados en robots.txt (índice → web y news)
    BASE + "/sitemap/web/{s}/?outputType=xml",
    BASE + "/sitemap/news/{s}/?outputType=xml",
]
pd.DataFrame(SECCION_EC.items(), columns=["categoria", "seccion El Comercio"])

,categoria,seccion El Comercio
0,politica,politica
1,economia,economia
2,deportes,deporte-total
3,ciencia_tecnologia,tecnologia


## 6. Extracción

### 6.1 RSS oficiales
Cada feed trae directamente titular, bajada (`description`), enlace y fecha. Se guarda el XML crudo de cada feed en `datos/raw/`.

In [8]:
def parsear_rss(xml):
    soup = BeautifulSoup(xml, "xml")
    filas = []
    for it in soup.find_all("item"):
        link = it.find("link").get_text(strip=True) if it.find("link") else ""
        desc = it.find("description").get_text() if it.find("description") else ""
        filas.append({
            "titular": it.find("title").get_text(strip=True) if it.find("title") else "",
            "bajada":  texto_limpio(desc),
            "medio":   MEDIO,
            "seccion_original": seccion_desde_url(link),
            "url":     link,
            "fecha":   it.find("pubDate").get_text(strip=True) if it.find("pubDate") else "",
            "fuente":  "rss",
        })
    return filas

filas, vistos = [], set()

def agregar(nuevas):
    """Agrega filas evitando URLs repetidas entre fuentes."""
    n = 0
    for f in nuevas:
        u = normalizar_url(f["url"])
        if f["url"] and u not in vistos:
            vistos.add(u); filas.append(f); n += 1
    return n

for cat, s in SECCION_EC.items():
    for i, patron in enumerate(FEEDS_RSS, start=1):
        url = patron.format(s=s)
        xml = descargar(url)
        if not xml or "<item" not in xml:
            print(f"  {cat:<20} feed {i}: sin datos")
            continue
        guardar_crudo(f"rss_{s}_{i}_{STAMP}.xml", xml)
        items = parsear_rss(xml)
        print(f"  {cat:<20} feed {i}: {len(items):3d} ítems, {agregar(items):3d} nuevos")

def contar_por_categoria():
    return pd.Series([categoria_de(f["seccion_original"]) for f in filas]).value_counts()

conteo = contar_por_categoria()
faltan = {c: max(0, META_POR_CATEGORIA - int(conteo.get(c, 0))) for c in CATEGORIAS}
print("\nObtenidas por RSS:", {c: int(conteo.get(c, 0)) for c in CATEGORIAS})
print("Faltan para la meta:", faltan)

  politica             feed 1:  23 ítems,  23 nuevos
  politica             feed 2:  23 ítems,   0 nuevos
  economia             feed 1:  25 ítems,  25 nuevos
  economia             feed 2:  25 ítems,   0 nuevos
  deportes             feed 1:  76 ítems,  76 nuevos
  deportes             feed 2:  76 ítems,   0 nuevos
  ciencia_tecnologia   feed 1:  17 ítems,  16 nuevos
  ciencia_tecnologia   feed 2:  17 ítems,   0 nuevos

Obtenidas por RSS: {'politica': 20, 'economia': 24, 'deportes': 74, 'ciencia_tecnologia': 16}
Faltan para la meta: {'politica': 100, 'economia': 96, 'deportes': 46, 'ciencia_tecnologia': 104}


### 6.2 Complemento con sitemaps de cada sección (solo lo que falte)
Los sitemaps solo traen la URL y la fecha. Por eso se visita cada artículo **una sola vez** para leer sus metadatos públicos (`og:title`, `og:description`, `article:published_time`), y únicamente hasta completar lo que falta en cada categoría.

⏱️ Con la pausa de 1.5 s, cada artículo tarda unos 2 s. Para unos 200 artículos son aproximadamente 7 minutos. Es intencional.

In [9]:
def leer_sitemap(xml):
    soup = BeautifulSoup(xml, "xml")
    entradas = []
    for nodo in soup.find_all("url"):
        loc = nodo.find("loc")
        if not loc:
            continue
        fecha  = nodo.find("publication_date") or nodo.find("lastmod")
        titulo = nodo.find("title")
        entradas.append({"loc":    loc.get_text(strip=True),
                         "fecha":  fecha.get_text(strip=True) if fecha else "",
                         "titulo": titulo.get_text(strip=True) if titulo else ""})
    return entradas

def meta(soup, *claves):
    for k in claves:
        tag = soup.find("meta", attrs={"property": k}) or soup.find("meta", attrs={"name": k})
        if tag and tag.get("content"):
            return tag["content"].strip()
    return ""

def parsear_articulo(html, url, fecha_sitemap="", titulo_sitemap=""):
    soup = BeautifulSoup(html, "html.parser")
    titular = meta(soup, "og:title") or titulo_sitemap or (soup.h1.get_text(strip=True) if soup.h1 else "")
    titular = re.sub(r"\s*\|\s*El Comercio.*$", "", titular, flags=re.I)   # quita "| El Comercio Perú"
    return {
        "titular": titular,
        "bajada":  meta(soup, "og:description", "description"),
        "medio":   MEDIO,
        "seccion_original": seccion_desde_url(url),
        "url":     url,
        "fecha":   meta(soup, "article:published_time") or fecha_sitemap,
        "fuente":  "sitemap+html",
    }

for cat, s in SECCION_EC.items():
    if faltan[cat] <= 0:
        print(f"{cat:<20} ✔ meta cumplida con RSS")
        continue

    # 1) Reunir URLs candidatas de los sitemaps de la sección
    candidatos = []
    for i, patron in enumerate(SITEMAPS_SECCION, start=1):
        url_sm = patron.format(s=s)
        xml = descargar(url_sm)
        if not xml:
            continue
        guardar_crudo(f"sitemap_{s}_{i}_{STAMP}.xml", xml)
        for e in leer_sitemap(xml):
            u = normalizar_url(e["loc"])
            if u not in vistos and categoria_de(seccion_desde_url(u)) == cat:
                candidatos.append(e); vistos.add(u)
    print(f"{cat:<20} faltan {faltan[cat]:3d} | candidatos en sitemaps: {len(candidatos)}")

    # 2) Visitar solo los artículos necesarios
    for e in candidatos:
        if faltan[cat] <= 0:
            break
        html = descargar(e["loc"])          # si un artículo falla, se pasa al siguiente
        if html:
            filas.append(parsear_articulo(html, e["loc"], e["fecha"], e["titulo"]))
            faltan[cat] -= 1
    print(f"{'':<20} → quedan por completar: {faltan[cat]}")

conteo = contar_por_categoria()
print("\nTotal recolectado por categoría:", {c: int(conteo.get(c, 0)) for c in CATEGORIAS})
if any(v > 0 for v in faltan.values()):
    print("⚠️ Alguna categoría no llegó a la meta: vuelve a ejecutar otro día; los respaldos se acumulan.")

politica             faltan 100 | candidatos en sitemaps: 72
                     → quedan por completar: 28
economia             faltan  96 | candidatos en sitemaps: 75
                     → quedan por completar: 21
deportes             faltan  46 | candidatos en sitemaps: 24
                     → quedan por completar: 22
ciencia_tecnologia   faltan 104 | candidatos en sitemaps: 79
                     → quedan por completar: 25

Total recolectado por categoría: {'politica': 92, 'economia': 99, 'deportes': 98, 'ciencia_tecnologia': 95}
⚠️ Alguna categoría no llegó a la meta: vuelve a ejecutar otro día; los respaldos se acumulan.


## 7. Respaldo crudo de esta ejecución

In [12]:
df_ejecucion = pd.DataFrame(filas)
ruta = f"datos/raw/crudo_{STAMP}.csv"
df_ejecucion.to_csv(ruta, index=False, encoding="utf-8")
print(ruta, df_ejecucion.shape)
df_ejecucion["fuente"].value_counts()

datos/raw/crudo_20261018_0004.csv (390, 7)


fuente
sitemap+html    250
rss             140
Name: count, dtype: int64

## 8. Limpieza y normalización

La limpieza se hace en **dos partes**:

| Parte | Qué hace | Descartes que registra |
|---|---|---|
| **8.1 Limpieza de datos** | Une todos los respaldos crudos y normaliza textos, URLs y fechas | Titular vacío, URL de otro dominio, URL duplicada, fecha no interpretable |
| **8.2 Mapeo y balanceo** | Asigna cada `seccion_original` a una de las 4 categorías, muestra la tabla de equivalencias y balancea | Sección fuera de las 4 categorías, exceso por balanceo |

Cada descarte queda registrado con su motivo en la lista `descartes`, y al final se muestra el reporte completo.

### 8.1 Limpieza de datos
Se unen los respaldos `datos/raw/crudo_*.csv` de **todas** las fechas de descarga. Luego se normalizan los textos, se dejan las URLs sin `?parámetros` ni `#anclas` (para detectar duplicados reales) y se convierten las fechas del RSS y de los sitemaps al formato `AAAA-MM-DD`, en hora de Lima.

In [13]:
COLUMNAS_CRUDAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha"]

# ── 1. Unir respaldos crudos ───────────────────────────────────────────────
archivos = sorted(glob.glob("datos/raw/crudo_*.csv"))
if not archivos:
    raise FileNotFoundError("No hay respaldos en datos/raw/. Ejecuta primero las secciones 5 a 7.")

df = pd.concat([pd.read_csv(a, dtype=str, keep_default_na=False) for a in archivos],
               ignore_index=True)

faltantes = set(COLUMNAS_CRUDAS) - set(df.columns)
if faltantes:
    raise ValueError(f"Faltan columnas en los respaldos: {faltantes}")

n_inicial = len(df)
descartes = []      # registro de cada descarte: motivo y cantidad

def descartar(df, mascara, motivo):
    """Quita las filas donde `mascara` es True y registra el motivo."""
    n = int(mascara.sum())
    descartes.append({"motivo": motivo, "n_descartadas": n})
    print(f"  {motivo:<42} -{n:>4}  → quedan {len(df) - n}")
    return df.loc[~mascara].copy()

print(f"{len(archivos)} respaldo(s) unidos → {n_inicial} filas iniciales")
for a in archivos:
    print("   ", a)

# ── 2. Normalizar textos, URLs y fechas ────────────────────────────────────
for col in ["titular", "bajada", "seccion_original"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()

df["url"] = df["url"].map(normalizar_url)

fechas = pd.to_datetime(df["fecha"], errors="coerce", utc=True, format="mixed")
df["fecha"] = fechas.dt.tz_convert("America/Lima").dt.strftime("%Y-%m-%d")

# ── 3. Descartes de limpieza ───────────────────────────────────────────────
print("\nDescartes de limpieza:")
df = descartar(df, df["titular"].eq(""), "Titular vacío")
df = descartar(df, ~df["url"].str.match(r"https?://(?:www\.)?elcomercio\.pe/"), "URL de otro dominio")
df = descartar(df, df.duplicated(subset="url", keep="first"), "URL duplicada")
df = descartar(df, df["fecha"].isna(), "Fecha no interpretable")

print(f"\nFilas limpias: {len(df)} | rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")

3 respaldo(s) unidos → 845 filas iniciales
    datos/raw\crudo_20261017_2211.csv
    datos/raw\crudo_20261017_2315.csv
    datos/raw\crudo_20261018_0004.csv

Descartes de limpieza:
  Titular vacío                              -   0  → quedan 845
  URL de otro dominio                        -   0  → quedan 845
  URL duplicada                              - 450  → quedan 395
  Fecha no interpretable                     -   0  → quedan 395

Filas limpias: 395 | rango de fechas: 2026-10-06 a 2026-10-18


### 8.2 Mapeo a categorías y balanceo
La tabla de equivalencias se construye **antes** de descartar, para mostrar también las secciones rechazadas y por qué no entran (no se crea una quinta categoría). Después se balancea con un máximo de `MAX_POR_CATEGORIA` noticias por categoría, elegidas al azar con `random_state=42` para que el resultado sea reproducible.

In [14]:
# ── 1. Tabla de equivalencias seccion_original → etiqueta ──────────────────
df["etiqueta"] = df["seccion_original"].map(categoria_de)

tabla_mapeo = (df.groupby("seccion_original", as_index=False)
                 .size()
                 .rename(columns={"size": "n_noticias"}))
tabla_mapeo["etiqueta"] = (tabla_mapeo["seccion_original"].map(categoria_de)
                             .fillna("❌ descartada (no encaja)"))
tabla_mapeo = (tabla_mapeo.sort_values(["etiqueta", "n_noticias"], ascending=[True, False])
                          .reset_index(drop=True))

pd.set_option("display.max_rows", 200)
print("Tabla de equivalencias:")
display(tabla_mapeo)

# ── 2. Descartes de mapeo y balanceo ───────────────────────────────────────
print("\nDescartes de mapeo y balanceo:")
df = descartar(df, df["etiqueta"].isna(), "Sección fuera de las 4 categorías")

conservar = (df.sample(frac=1, random_state=42)
               .groupby("etiqueta")
               .head(MAX_POR_CATEGORIA)
               .index)
df = descartar(df, ~df.index.isin(conservar), f"Exceso sobre {MAX_POR_CATEGORIA} por categoría")
df = df.sort_values(["etiqueta", "fecha"]).reset_index(drop=True)

# ── 3. Reporte final de descartes ──────────────────────────────────────────
reporte = pd.DataFrame(descartes)
reporte.loc[len(reporte)] = {"motivo": "TOTAL descartadas", "n_descartadas": n_inicial - len(df)}

print(f"\nFilas iniciales: {n_inicial} → filas finales: {len(df)}")
print(df["etiqueta"].value_counts().to_string())
if len(df) < 350:
    print("\n⚠️ Menos de 350 noticias: ejecuta la recolección otro día (los respaldos se acumulan).")

print("\nReporte de descartes:")
display(reporte)

Tabla de equivalencias:


,seccion_original,n_noticias,etiqueta
0,tecnologia-actualidad,37,ciencia_tecnologia
1,tecnologia-inteligencia-artificial,19,ciencia_tecnologia
2,tecnologia-ciencias,10,ciencia_tecnologia
3,tecnologia-ciberseguridad,7,ciencia_tecnologia
4,tecnologia-ecologia,6,ciencia_tecnologia
5,tecnologia-videojuegos,5,ciencia_tecnologia
6,tecnologia-moviles,4,ciencia_tecnologia
7,tecnologia-gadgets,3,ciencia_tecnologia
8,tecnologia-redes-sociales,2,ciencia_tecnologia
9,tecnologia-robotica,1,ciencia_tecnologia



Descartes de mapeo y balanceo:
  Sección fuera de las 4 categorías          -   6  → quedan 389
  Exceso sobre 100 por categoría             -   1  → quedan 388

Filas iniciales: 845 → filas finales: 388
etiqueta
deportes              100
economia               99
ciencia_tecnologia     95
politica               94

Reporte de descartes:


,motivo,n_descartadas
0,Titular vacío,0
1,URL de otro dominio,0
2,URL duplicada,450
3,Fecha no interpretable,0
4,Sección fuera de las 4 categorías,6
5,Exceso sobre 100 por categoría,1
6,TOTAL descartadas,457


### 8.3 Verificación de URLs funcionales (muestra de 15)

In [15]:
muestra = df.sample(15, random_state=42)["url"]
estado = {}
for u in muestra:
    try:
        time.sleep(segundos_de_pausa)
        estado[u] = requests.head(u, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True).status_code
    except requests.RequestException as e:
        estado[u] = type(e).__name__
verif = pd.Series(estado, name="HTTP").to_frame()
print(f"URLs con HTTP 200: {(verif['HTTP'] == 200).sum()}/15")
verif

URLs con HTTP 200: 15/15


,HTTP
https://elcomercio.pe/economia/mercados/precio-del-dolar-en-peru-2026-a-cuanto-esta-el-dolar-este-viernes-16-de-octubre-tipo-de-cambio-segun-el-bcrp-tasa-cotizacion-compra-y-venta-segun-bcrp-dolares-a-soles-sbs-ocona-lbposting-noticia/,200
https://elcomercio.pe/tecnologia/actualidad/ben-affleck-se-revela-como-un-entusiasta-y-experto-en-inteligencia-artificial-superinteligencia-cine-ia-noticia/,200
https://elcomercio.pe/deporte-total/futbol-peruano/video-liga-1-max-en-vivo-gratis-futbol-libre-tv-universitario-vs-alianza-atletico-hoy-por-internet-via-liga-1-play-dgo-por-torneo-clausura-liga-1-noticia/,200
https://elcomercio.pe/tecnologia/inteligencia-artificial/microsoft-y-nvidia-impulsan-una-nueva-categoria-de-computadoras-para-la-ia-con-el-superchip-rtx-spark-agentes-de-ia-asus-hp-lenovo-dell-noticia/,200
https://elcomercio.pe/politica/actualidad/papa-leon-xiv-keiko-fujimori-esperamos-que-su-visita-se-pueda-dar-pero-entenderemos-tambien-el-caso-contrario-apuesplan-ultimas-noticia/,200
https://elcomercio.pe/deporte-total/futbol-mundial/partidos-de-hoy-sabado-17-de-octubre-2026-en-vivo-liga-1-laliga-horarios-canales-tv-y-donde-ver-futbol-online-gratis-noticia/,200
https://elcomercio.pe/deporte-total/futbol-peruano/tabla-de-posiciones-liga-1-en-vivo-torneo-clausura-acumulado-resultados-de-la-fecha-12-noticia/,200
https://elcomercio.pe/tecnologia/ciencias/dia-mundial-del-huevo-solo-el-4-de-las-gallinas-utilizadas-en-la-industria-peruana-vive-libre-de-jaulas-noticia/,200
https://elcomercio.pe/deporte-total/polideportivo/ignacio-buse-hare-parte-de-pretemporada-en-lima-para-ir-directo-al-australian-open-entrevista-noticia/,200
https://elcomercio.pe/deporte-total/futbol-peruano/gol-de-jairo-velez-hoy-en-el-alianza-lima-vs-atletico-grau-por-liga-1-video-noticia/,200


## 9. Análisis exploratorio (EDA)
Los títulos se calculan a partir de los datos para que expresen la conclusión. **Revísalos** y ajústalos si tu dataset muestra algo distinto.

In [16]:
df["len_titular"] = df["titular"].str.len()

dist = df["etiqueta"].value_counts().reset_index()
dist.columns = ["etiqueta", "n"]
titulo1 = (f"Dataset balanceado: entre {dist.n.min()} y {dist.n.max()} noticias por categoría "
           f"({dist.iloc[0].etiqueta} es la mayor)")
fig1 = px.bar(dist, x="etiqueta", y="n", color="etiqueta", text="n", title=titulo1,
              labels={"etiqueta": "Categoría", "n": "N.º de noticias"})
mostrar(fig1)

⚠️ No se pudo generar PNG (¿kaleido instalado?): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [17]:
por_dia = df.groupby(["fecha", "etiqueta"]).size().reset_index(name="n")
total_dia = df.groupby("fecha").size()
fmin, fmax = df["fecha"].min(), df["fecha"].max()
titulo2 = (f"Las noticias cubren {df['fecha'].nunique()} días ({fmin} a {fmax}); "
           f"el pico fue el {total_dia.idxmax()} con {total_dia.max()} noticias")
fig2 = px.bar(por_dia, x="fecha", y="n", color="etiqueta", title=titulo2,
              labels={"fecha": "Fecha de publicación", "n": "N.º de noticias", "etiqueta": "Categoría"})
mostrar(fig2)

⚠️ No se pudo generar PNG (¿kaleido instalado?): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [18]:
prom = df.groupby("etiqueta")["len_titular"].mean().sort_values(ascending=False)
titulo3 = (f"Los titulares miden {df.len_titular.mean():.0f} caracteres en promedio; "
           f"{prom.index[0]} usa los más largos ({prom.iloc[0]:.0f}) y {prom.index[-1]} los más cortos ({prom.iloc[-1]:.0f})")
fig3 = px.histogram(df, x="len_titular", color="etiqueta", nbins=30, barmode="overlay", opacity=0.6,
                    title=titulo3, labels={"len_titular": "Longitud del titular (caracteres)", "etiqueta": "Categoría"})
mostrar(fig3)

⚠️ No se pudo generar PNG (¿kaleido instalado?): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## 10. Validación: titulares más largos y más cortos

In [19]:
cols_v = ["titular", "len_titular", "etiqueta"]
print("Rango en el histograma:", df.len_titular.min(), "–", df.len_titular.max(), "caracteres\n")
print("🔝 5 titulares MÁS LARGOS"); display(df.nlargest(5, "len_titular")[cols_v])
print("🔻 5 titulares MÁS CORTOS"); display(df.nsmallest(5, "len_titular")[cols_v])

Rango en el histograma: 22 – 178 caracteres

🔝 5 titulares MÁS LARGOS


,titular,len_titular,etiqueta
193,“El jugador peruano tiene un déficit grande en...,178,deportes
146,“Con Mano Menezes el mejor jugador de la selec...,171,deportes
192,El furioso “Que se dedique a administrar” de G...,171,deportes
190,“Creo que puedo llegar a un top 3 sin que nadi...,167,deportes
345,Keiko Fujimori sobre Álvaro Vargas Llosa: “Con...,166,politica


🔻 5 titulares MÁS CORTOS


,titular,len_titular,etiqueta
270,Invocación por el Perú,22,economia
290,AlaCena no tiene precio,23,economia
286,IA y Sandbox regulatorio,24,economia
230,La evolución de la confianza en los pagos,41,economia
293,Premio Nobel de Economía: la macro es micro,43,economia


> **Comentario (completar):** los valores extremos coinciden con las colas del histograma. Los más cortos suelen ser … y los más largos …

## 11. Exportación final

In [20]:
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
df_final = df[COLUMNAS].reset_index(drop=True)
df_final.to_csv("datos/noticias.csv", index=False, encoding="utf-8")
print("df.shape:", df_final.shape)
df_final.head()

df.shape: (388, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,¿Cómo lograr que una marca aparezca en ChatGPT...,"Cuando un usuario le pregunta a ChatGPT, Gemin...",El Comercio,tecnologia-inteligencia-artificial,https://elcomercio.pe/tecnologia/inteligencia-...,2026-10-06,ciencia_tecnologia
1,Finlandia frena la construcción de dos centros...,La inversión sumaba 14.600 millones de dólares...,El Comercio,tecnologia-actualidad,https://elcomercio.pe/tecnologia/actualidad/fi...,2026-10-06,ciencia_tecnologia
2,Noruega estudia prohibir temporalmente las gaf...,La prohibición busca evitar la captura de imág...,El Comercio,tecnologia-gadgets,https://elcomercio.pe/tecnologia/gadgets/norue...,2026-10-06,ciencia_tecnologia
3,Investigan caso en Siberia: ¿qué sabemos de la...,"Aunque suele asociarse con el pasado, la peste...",El Comercio,tecnologia-actualidad,https://elcomercio.pe/tecnologia/actualidad/in...,2026-10-06,ciencia_tecnologia
4,"EE.UU. ""sigue de cerca"" el caso de una técnica...","El secretario de Estado, Marco Rubio, expresó ...",El Comercio,tecnologia-actualidad,https://elcomercio.pe/tecnologia/actualidad/ee...,2026-10-06,ciencia_tecnologia
